# 1. LangChain 도구와 LangGraph 루프

근거: [설계 글 §1·§4.1](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 모델의 판단과 harness의 책임
모델은 다음 행동을 제안하고, harness는 도구 실행·상태 기록·중단·결과 전달을 관리합니다. 아래는 실제 LangChain 도구와 LangGraph ToolNode를 사용하는 최소 루프입니다. 모델 부분만 결정적인 Runnable로 대체해 API 키 없이 흐름을 관찰합니다. 자연어 이해 성능을 평가하는 예제가 아닙니다.

흐름: START → model → (도구 요청이면 tools → model, 아니면 END). add_messages reducer는 노드가 반환한 새 메시지를 기존 대화에 합칩니다.

In [1]:
from typing import Annotated, TypedDict
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, AnyMessage
from langchain_core.runnables import RunnableLambda
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

@tool
def lookup(topic: str) -> str:
    """로컬 학습 문서에서 개념의 요약을 찾습니다."""
    return {'harness':'모델을 둘러싼 실행·상태·정책 시스템'}.get(topic, '자료 없음')

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def decide(messages):
    if isinstance(messages[-1], ToolMessage):
        return AIMessage(content='조회 결과: '+messages[-1].content)
    return AIMessage(content='',tool_calls=[{'name':'lookup','args':{'topic':'harness'},'id':'lookup-1','type':'tool_call'}])

model=RunnableLambda(decide)
def model_node(s): return {'messages':[model.invoke(s['messages'])]}
def route(s): return 'tools' if s['messages'][-1].tool_calls else 'done'
b=StateGraph(State)
b.add_node('model',model_node)
b.add_node('tools',ToolNode([lookup]))
b.add_edge(START,'model')
b.add_conditional_edges('model',route,{'tools':'tools','done':END})
b.add_edge('tools','model')
app=b.compile()
result=app.invoke({'messages':[HumanMessage(content='harness 설명해줘')]}, {'recursion_limit':8})
for m in result['messages']: print(m.type, m.content or m.tool_calls)
assert len(result['messages'])==4
assert isinstance(result['messages'][2],ToolMessage)
assert '실행·상태·정책' in result['messages'][-1].content

human harness 설명해줘
ai [{'name': 'lookup', 'args': {'topic': 'harness'}, 'id': 'lookup-1', 'type': 'tool_call'}]
tool 모델을 둘러싼 실행·상태·정책 시스템
ai 조회 결과: 모델을 둘러싼 실행·상태·정책 시스템


## 관찰
모델 호출은 두 번이지만 실제 도구 실행은 한 번입니다. ToolMessage의 tool_call_id가 AIMessage의 요청과 결과를 연결합니다. 이 루프 안에서 오래 걸리는 도구를 동기 실행하면 다음 입력 처리가 늦어집니다. 단순히 노드를 async def로 바꾸는 것만으로 별도 입력 처리가 자동 구현되지는 않습니다.

In [2]:
assert result['messages'][2].tool_call_id == result['messages'][1].tool_calls[0]['id']
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools<hr/><small><em>__tools = lookup</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -. &nbsp;done&nbsp; .-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 직접 바꿔 보기

lookup의 topic을 존재하지 않는 값으로 바꾸어 실행하세요. 결과 없음은 모델 실패인가요, 도구의 정상 결과인가요? 무한 도구 호출을 막는 예산은 어디에 둘까요?

## 해설과 참고 답안

자료 없음은 정상적인 도구 결과입니다. 모델이 계속 도구를 요구하면 graph recursion_limit 또는 별도 step budget으로 중단합니다. 예외를 성공 메시지로 숨기지 말고 결과 없음·실행 오류·예산 초과를 구분해야 합니다.